# Teks + angka — EDA lengkap (kenali data sebelum memilih model)
**Kapan dipakai:** 30 menit pertama, untuk data teks dengan atau tanpa kolom angka/kategori.
Target boleh kelas (klasifikasi) atau angka (regresi): target angka dipotong jadi 4 kelompok
(Q1 rendah … Q4 tinggi) supaya semua grafik "per kelompok target" tetap jalan.

**Alur (Run All, baca dari atas; tiap temuan penting dicetak dengan tanda `-` dan masuk
`temuan.md`):**

| bagian | pertanyaan yang dijawab |
|---|---|
| 1 | kolom apa saja, tipenya apa, mana yang kosong, mana yang hanya ada di train |
| 2 | target: seimbang atau timpang, sebarannya |
| 3 | nilai kosong: di mana, dan apakah "kosong" itu sendiri berhubungan dengan target |
| 4 | kolom angka: sebaran, pencilan, train vs test, per kelompok target |
| 5 | kolom kategori: jumlah nilai, nilai langka, nilai yang hanya ada di test, vs target |
| 6 | korelasi antarkolom angka (kolom kembar) |
| 7 | teks: panjang, ciri (URL, angka, kapital), kata khas, frasa, kata baru di test, kembar |
| 8 | **kekuatan tiap kolom sendirian terhadap target** (id dan urutan baris ikut = cek bocor) |
| 9 | train vs test: kolom mana yang bergeser, teks test beda gaya atau tidak |
| 10 | teks ↔ angka: korelasi ciri teks dengan kolom angka |
| 11 | **sinyal ada di mana: teks saja vs tabel saja** (menentukan perlu stacking atau tidak) |
| 12 | waktu (hanya bila `DATE_COL` diisi) |

**Hasil:** gambar `.png`, `profil_kolom.csv`, `kekuatan_kolom.csv`, `temuan.md` di
`OUTPUT_FOLDER`.
Notebook ini hanya membaca, tidak mengubah data. Membersihkan: `praproses_lengkap`. Gambar
kualitas slide khusus teks: `eda_presentasi`.

**Saat lomba:** ubah sel Konfigurasi, Run All. **Data demo:** ulasan baju (2 kolom teks, kolom
angka dan kategori, target `rating` 1–5).

In [ ]:
import os
import warnings

import emoji
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"       # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"            # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"              # GANTI saat lomba
# TRAIN_CSV = "../../../outputs/text/praproses_lengkap/train_bersih.csv"   # hasil praproses_lengkap
# TEST_CSV = "../../../outputs/text/praproses_lengkap/test_bersih.csv"
OUTPUT_FOLDER = "../../../outputs/text/eda_lengkap"

ID_COL = "id"                          # GANTI saat lomba
TEXT_COLS = ["title", "review_text"]   # GANTI saat lomba: satu atau beberapa kolom teks
TARGET = "rating"                      # GANTI saat lomba
DATE_COL = None                        # GANTI: kolom tanggal (bagian 12), None = tidak ada
LANGUAGE = "en"                        # GANTI: "id" = stopword Indonesia (Sastrawi), "en" = Inggris
CAT_MAX_UNIQUE = 20                    # kolom angka dengan <= sekian nilai unik dianggap kategori
SAMPLE_ROWS = 5000                     # baris untuk sel yang melatih model kecil (bagian 9 dan 11)
AMAN_NDA = True                        # True = tidak mencetak teks asli / baris data,
#                                        luaran aman dibawa keluar
RARE_MIN = 2                           # kata khas train/test harus muncul di >= sekian teks
if AMAN_NDA:
    RARE_MIN = 5
SEED = 42

## Alat kecil
`note` mencatat temuan, `show` menyimpan gambar, `grid` membuat kisi grafik kecil.

In [ ]:
plt.rcParams.update({
    "font.size": 12, "axes.titlesize": 13, "axes.spines.top": False, "axes.spines.right": False,
    "axes.axisbelow": True, "axes.grid": True, "grid.linestyle": "--", "grid.linewidth": 0.6,
    "grid.color": "#d9d9d9", "figure.dpi": 70, "savefig.dpi": 150, "savefig.bbox": "tight",
})
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#7F7F7F", "#F0E442"]
findings = []


def note(text):
    findings.append(text)
    print("-", text)


def show(figure, name):
    figure.tight_layout()
    figure.savefig(f"{OUTPUT_FOLDER}/{name}.png")
    plt.show()


def share(mask):
    return round(100 * mask.mean(), 1)


def grid(count, columns=4):
    rows = int(np.ceil(count / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(13, 3.2 * rows), squeeze=False)
    for ax in axes.ravel()[count:]:
        ax.axis("off")
    return figure, axes.ravel()

## 1. Baca data, jenis kolom, profil
Jenis kolom ditebak: angka dengan banyak nilai unik → `NUM_COLS`; angka dengan sedikit nilai unik
dan teks pendek yang berulang → `CAT_COLS`; teks yang hampir semuanya unik tapi tidak ada di
`TEXT_COLS` → `FREE_COLS` (pertimbangkan masuk `TEXT_COLS`). Tebakan salah (mis. kode produk
terbaca sebagai angka): tulis daftarnya sendiri di baris `# GANTI`.

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
target_missing = int(train[TARGET].isna().sum())
train = train[train[TARGET].notna()].reset_index(drop=True)   # baris tanpa target dibuang
only_train = []
for column in train.columns:
    if column not in test.columns:
        only_train.append(column)
print("train:", train.shape, "| test:", test.shape)
print("kolom hanya di train (target / bocoran, jangan jadi fitur):", only_train)

NUM_COLS, CAT_COLS, FREE_COLS = [], [], []
for column in train.columns:
    if column in [ID_COL, TARGET, DATE_COL] + TEXT_COLS or column not in test.columns:
        continue
    is_number = pd.api.types.is_numeric_dtype(train[column])
    if is_number and train[column].nunique() > CAT_MAX_UNIQUE:
        NUM_COLS.append(column)
    elif is_number or train[column].nunique() <= 0.5 * len(train):
        CAT_COLS.append(column)
    else:
        FREE_COLS.append(column)
# NUM_COLS = ["age", "positive_feedback_count"]      # GANTI bila tebakan salah
# CAT_COLS = ["clothing_id", "division_name"]        # GANTI bila tebakan salah
print("angka:", NUM_COLS, "\nkategori:", CAT_COLS, "\nteks bebas lain:", FREE_COLS)
if not AMAN_NDA:
    print(train.head(3))

In [ ]:
def join_text(table):
    joined = table[TEXT_COLS[0]].fillna("").astype(str)
    for column in TEXT_COLS[1:]:
        joined = joined + " " + table[column].fillna("").astype(str)
    return joined.str.strip()


def profile_row(column):
    values = train[column]
    row = {"kolom": column, "tipe": str(values.dtype), "kosong train %": share(values.isna()),
           "kosong test %": np.nan, "unik": values.nunique(),
           "contoh": str(values.dropna().iloc[0])[:40 * (not AMAN_NDA)]}   # kosong bila NDA
    if column in test.columns:
        row["kosong test %"] = share(test[column].isna())
    return row


raw_train = join_text(train)
raw_test = join_text(test)
profile_rows = []
for column in train.columns:
    profile_rows.append(profile_row(column))
profile = pd.DataFrame(profile_rows)
profile.to_csv(f"{OUTPUT_FOLDER}/profil_kolom.csv", index=False)
constant = profile.loc[profile["unik"] <= 1, "kolom"].tolist()
note(f"data: train {len(train)} baris, test {len(test)} baris, {train.shape[1]} kolom")
if constant:
    note(f"kolom berisi satu nilai saja (buang): {constant}")
note(f"id kembar: train {train[ID_COL].duplicated().sum()}, test {test[ID_COL].duplicated().sum()}")
profile

## 2. Target
`group` = kelompok target yang dipakai semua grafik berikutnya (kelas asli, atau 4 kuartil bila
target angka). Timpang (kelas terkecil jauh lebih sedikit): pakai `class_weight="balanced"`, split
berstrata, dan metrik makro; jangan percaya akurasi.

In [ ]:
target_is_number = pd.api.types.is_numeric_dtype(train[TARGET])
target_is_number = target_is_number and train[TARGET].nunique() > CAT_MAX_UNIQUE
if target_is_number:
    quartile = pd.qcut(train[TARGET], 4, labels=False, duplicates="drop")
    group = "Q" + (quartile + 1).astype(int).astype(str)
else:
    group = train[TARGET].astype(str)
group_counts = group.value_counts()
shown_groups = sorted(group_counts.index[:8])     # paling banyak 8 kelompok di grafik

figure, ax = plt.subplots(figsize=(13, 3.6))
if target_is_number:
    train[TARGET].hist(bins=50, ax=ax, color=PALETTE[0])
    note(f"target angka: median {train[TARGET].median():.3g}, "
         f"rata-rata {train[TARGET].mean():.3g}, skew {train[TARGET].skew():.2f} "
         f"(|skew| > 1: coba log1p), nol {share(train[TARGET] == 0)}%")
else:
    group_counts.sort_index().plot.bar(ax=ax, color=PALETTE[0], rot=0)
    ratio = group_counts.iloc[0] / group_counts.iloc[-1]
    note(f"target {len(group_counts)} kelas: terbanyak {group_counts.index[0]} "
         f"({share(group == group_counts.index[0])}%), tersedikit {group_counts.index[-1]} "
         f"({group_counts.iloc[-1]} baris), rasio {ratio:.1f}x")
ax.set_xlabel(TARGET)
note(f"target kosong: {target_missing} baris (dibuang dari EDA ini)")
show(figure, "02_target")

## 3. Nilai kosong
`p kosong vs target` kecil (< 0,01) = baris yang kosong punya sebaran target berbeda: jangan diisi
diam-diam, tambahkan kolom penanda `<kolom>_kosong` (atau biarkan NaN untuk LightGBM).

In [ ]:
missing_rows = []
for column in train.columns:
    is_missing = train[column].isna()
    if column == TARGET or is_missing.sum() == 0:
        continue
    row = {"kolom": column, "kosong train %": share(is_missing), "kosong test %": np.nan,
           "p kosong vs target": np.nan}
    if column in test.columns:
        row["kosong test %"] = share(test[column].isna())
    if is_missing.sum() >= 20:
        row["p kosong vs target"] = stats.chi2_contingency(pd.crosstab(is_missing, group))[1]
    missing_rows.append(row)
missing_table = pd.DataFrame(missing_rows)
if len(missing_table) == 0:
    note("tidak ada nilai kosong di train")
else:
    informative = missing_table.loc[missing_table["p kosong vs target"] < 0.01, "kolom"].tolist()
    note(f"kolom dengan nilai kosong: {missing_table['kolom'].tolist()}")
    note(f"kosongnya berhubungan dengan target (beri kolom penanda): {informative}")
missing_table

## 4. Kolom angka
- `skew` besar (> 1) dan semua nilai ≥ 0 → `log1p` untuk model linear/NN (pohon tidak butuh).
- `pencilan %` = di luar 1,5 × IQR. Pencilan belum tentu salah: lihat nilainya dulu.
- `KS vs test` > 0,1 = sebaran kolom itu beda antara train dan test.

In [ ]:
number_rows = []
for column in NUM_COLS:
    values = train[column].dropna()
    low, high = values.quantile(0.25), values.quantile(0.75)
    fence = 1.5 * (high - low)
    outside = (values < low - fence) | (values > high + fence)
    number_rows.append({
        "kolom": column, "median": values.median(), "rata-rata": round(values.mean(), 2),
        "min": values.min(), "maks": values.max(), "skew": round(values.skew(), 2),
        "nol %": share(values == 0), "negatif %": share(values < 0), "pencilan %": share(outside),
        "KS vs test": round(stats.ks_2samp(values, test[column].dropna())[0], 3)})
number_table = pd.DataFrame(number_rows)
if len(number_table) > 0:
    skewed = number_table.loc[number_table["skew"].abs() > 1, "kolom"].tolist()
    note(f"kolom angka: {NUM_COLS} | miring (|skew| > 1): {skewed}")
number_table

In [ ]:
# Sebaran tiap kolom angka: train (biru) vs test (jingga), dipotong di persentil 1-99
if NUM_COLS:
    figure, axes = grid(len(NUM_COLS))
    for ax, column in zip(axes, NUM_COLS):
        low, high = train[column].quantile(0.01), train[column].quantile(0.99)
        bins = np.linspace(low, max(high, low + 1e-9), 31)
        ax.hist(train[column].dropna(), bins=bins, density=True, alpha=0.6, color=PALETTE[0])
        ax.hist(test[column].dropna(), bins=bins, density=True, alpha=0.6, color=PALETTE[1])
        ax.set_title(column)
    figure.legend(["train", "test"], loc="upper right")
    show(figure, "04_angka_sebaran")

In [ ]:
# Tiap kolom angka per kelompok target: kotak yang bergeser antar kelompok = kolom berguna
if NUM_COLS:
    figure, axes = grid(len(NUM_COLS))
    for ax, column in zip(axes, NUM_COLS):
        boxes = []
        for name in shown_groups:
            boxes.append(train.loc[group == name, column].dropna())
        ax.boxplot(boxes, showfliers=False)
        ax.set_xticklabels(shown_groups)
        ax.set_title(column)
    show(figure, "04_angka_per_target")

## 5. Kolom kategori
- `langka` = nilai yang muncul < 10 kali (gabung jadi "lainnya" atau biarkan untuk model pohon).
- `test tak dikenal %` = baris test yang nilainya tidak pernah muncul di train: model tidak tahu
  apa-apa tentang baris itu; kalau besar, jangan andalkan kolom tersebut.
- `unik` sangat banyak (ratusan+) → target encoding DI DALAM fold atau CatBoost, bukan one-hot.

In [ ]:
category_rows = []
for column in CAT_COLS:
    counts = train[column].value_counts()
    unknown = ~test[column].isin(train[column]) & test[column].notna()
    category_rows.append({
        "kolom": column, "unik": len(counts), "terbanyak": str(counts.index[0])[:25],
        "porsi terbanyak %": round(100 * counts.iloc[0] / len(train), 1),
        "langka": int((counts < 10).sum()), "test tak dikenal %": share(unknown)})
category_table = pd.DataFrame(category_rows)
if len(category_table) > 0:
    unseen = category_table.loc[category_table["test tak dikenal %"] > 1, "kolom"].tolist()
    note(f"kolom kategori: {CAT_COLS} | punya nilai test yang tak ada di train (> 1%): {unseen}")
category_table

In [ ]:
# Porsi kelompok target di tiap nilai kategori (8 nilai tersering): batang yang komposisinya
# beda-beda = kategori itu berguna
if CAT_COLS:
    figure, axes = grid(len(CAT_COLS), columns=3)
    for ax, column in zip(axes, CAT_COLS):
        top_values = train[column].value_counts().index[:8]
        mix = pd.crosstab(train[column], group, normalize="index").loc[top_values]
        mix.plot.barh(stacked=True, ax=ax, legend=False, color=PALETTE, width=0.8)
        ax.set_title(column)
        ax.set_ylabel("")
        ax.invert_yaxis()
    figure.legend(mix.columns.tolist(), loc="lower right", ncol=4, title=TARGET)
    show(figure, "05_kategori_per_target")

## 6. Korelasi antarkolom angka (Spearman)
Dua kolom dengan |r| > 0,9 membawa informasi yang sama: salah satunya boleh dibuang. Kolom yang
|r|-nya terhadap target mendekati 1 = curiga bocor (lihat bagian 8).

In [ ]:
corr_cols = []
for column in NUM_COLS + CAT_COLS + [TARGET]:
    if pd.api.types.is_numeric_dtype(train[column]):
        corr_cols.append(column)


def heatmap(ax, table):
    ax.imshow(table.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=30, ha="right")
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.grid(False)
    for row in range(table.shape[0]):
        for column in range(table.shape[1]):
            value = table.iloc[row, column]
            color = "black"
            if abs(value) > 0.6:
                color = "white"
            ax.text(column, row, f"{value:.2f}", ha="center", va="center", color=color)


if len(corr_cols) >= 2:
    corr = train[corr_cols].corr(method="spearman")
    figure, ax = plt.subplots(figsize=(13, 0.6 * len(corr_cols) + 2))
    heatmap(ax, corr)
    show(figure, "06_korelasi")
    for first in range(len(corr_cols)):
        for second in range(first + 1, len(corr_cols)):
            if abs(corr.iloc[first, second]) > 0.9:
                note(f"korelasi tinggi: {corr_cols[first]} ~ {corr_cols[second]} "
                     f"(r = {corr.iloc[first, second]:.2f})")

## 7. Teks
### 7a. Tiap kolom teks: kosong, panjang, keunikan
Median kata sangat kecil (judul) → TF-IDF karakter membantu. `p95` > 400 kata → BERT memotong
(`contoh/text/dokumen_panjang`). `unik %` rendah → banyak teks kembar atau kolom itu sebenarnya
kategori.

In [ ]:
text_rows = []
for column in TEXT_COLS:
    for name, table in [("train", train), ("test", test)]:
        texts = table[column].fillna("").astype(str)
        words = texts.str.split().str.len()
        text_rows.append({"kolom": column, "bagian": name,
                          "kosong %": share(texts.str.strip() == ""),
                          "median kata": words.median(), "p95 kata": words.quantile(0.95),
                          "maks kata": words.max(),
                          "unik %": round(100 * texts.nunique() / len(texts), 1)})
text_table = pd.DataFrame(text_rows)
train_words = raw_train.str.split().str.len()
test_words = raw_test.str.split().str.len()
note(f"panjang teks gabungan (kata): train median {train_words.median():.0f}, "
     f"p95 {train_words.quantile(0.95):.0f}, maks {train_words.max()} | "
     f"test median {test_words.median():.0f} | > 400 kata: {share(train_words > 400)}%")
text_table

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 3.8))
bins = np.linspace(0, max(train_words.quantile(0.99), 1), 41)
axes[0].hist(train_words, bins=bins, density=True, alpha=0.6, color=PALETTE[0], label="train")
axes[0].hist(test_words, bins=bins, density=True, alpha=0.6, color=PALETTE[1], label="test")
axes[0].set_xlabel("jumlah kata per teks")
axes[0].legend()
boxes = []
for name in shown_groups:
    boxes.append(train_words[group == name])
axes[1].boxplot(boxes, showfliers=False)
axes[1].set_xticklabels(shown_groups)
axes[1].set_xlabel(f"jumlah kata per kelompok {TARGET}")
show(figure, "07_panjang_teks")

### 7b. Ciri teks per kelompok target
Angka = % teks di kelompok itu yang punya ciri tersebut. `selisih` besar = ciri itu membedakan
kelompok: JANGAN dibuang saat membersihkan (jadikan token atau fitur `meta_*`).

In [ ]:
FEATURE_PATTERNS = {
    "URL": r"https?://\S+|www\.\S+", "mention @": r"@\w+", "hashtag #": r"#\w+",
    "angka": r"\d", "tanda seru !": r"!", "tanda tanya ?": r"\?",
    "kata HURUF BESAR": r"\b[A-Z]{3,}\b", "huruf berulang 3+": r"([a-zA-Z])\1{2,}",
    "tag HTML": r"<[a-zA-Z/][^<>]*>", "bukan ASCII (emoji, huruf rusak)": r"[^\x00-\x7F]",
}
feature_rows = []
for name, pattern in FEATURE_PATTERNS.items():
    has_feature = raw_train.str.contains(pattern, regex=True)
    row = {"ciri": name, "train %": share(has_feature),
           "test %": share(raw_test.str.contains(pattern, regex=True))}
    for group_name in shown_groups:
        row[group_name] = share(has_feature[group == group_name])
    feature_rows.append(row)
feature_table = pd.DataFrame(feature_rows)
per_group = feature_table[shown_groups]
feature_table["selisih"] = per_group.max(axis=1) - per_group.min(axis=1)
feature_table = feature_table.sort_values("selisih", ascending=False)
top_feature = feature_table.iloc[0]
note(f"ciri teks paling membedakan kelompok: {top_feature['ciri']} "
     f"(selisih {top_feature['selisih']:.1f} poin persen)")
feature_table

### 7c. Kata khas per kelompok target (weighted log-odds, Monroe dkk. 2008)
Lebih jujur dari frekuensi (kata umum tidak menang) dan dari chi² (kata langka tidak meledak).
z > 2 kira-kira bermakna. Kata negasi sengaja tidak dibuang dari stopword. Kata aneh di sini
(nama sumber, kode, tanda tangan) = calon bocoran → cek di `praproses_lengkap` bagian 2.

In [ ]:
NEGATIONS = {"tidak", "bukan", "jangan", "belum", "kurang", "tak", "gak", "nggak", "not", "no"}
stop_words = "english"
if LANGUAGE == "id":
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    stop_words = sorted(set(StopWordRemoverFactory().get_stop_words()) - NEGATIONS)


def log_odds_z(counts, in_class):
    prior = np.asarray(counts.sum(axis=0)).ravel() + 0.01
    own = np.asarray(counts[in_class].sum(axis=0)).ravel()
    rest = np.asarray(counts[~in_class].sum(axis=0)).ravel()
    own_odds = np.log((own + prior) / (own.sum() + prior.sum() - own - prior))
    rest_odds = np.log((rest + prior) / (rest.sum() + prior.sum() - rest - prior))
    return (own_odds - rest_odds) / np.sqrt(1 / (own + prior) + 1 / (rest + prior))


word_counter = CountVectorizer(token_pattern=r"(?u)\b[^\W\d_]{2,}\b", stop_words=stop_words,
                               ngram_range=(1, 2), min_df=5)
word_counts = word_counter.fit_transform(raw_train)
vocabulary = word_counter.get_feature_names_out()
word_rows = []
for name in shown_groups:
    z_values = log_odds_z(word_counts, (group == name).to_numpy())
    for index in np.argsort(-z_values)[:12]:
        word_rows.append({"kelompok": name, "kata": vocabulary[index],
                          "z": round(z_values[index], 1)})
words_table = pd.DataFrame(word_rows)
print("kosakata (min 5 teks):", len(vocabulary))

In [ ]:
figure, axes = grid(len(shown_groups), columns=min(len(shown_groups), 4))
for position, name in enumerate(shown_groups):
    part = words_table[words_table["kelompok"] == name]
    axes[position].barh(part["kata"], part["z"], color=PALETTE[position % 8])
    axes[position].invert_yaxis()
    axes[position].set_title(f"{TARGET} = {name}")
    note(f"kata khas {name}: {', '.join(part['kata'].head(8))}")
show(figure, "07_kata_khas")

### 7d. Frasa tersering dan kata baru di test
Frasa tersering yang bukan isi (salam, tanda tangan, nama sumber) = boilerplate → buang di
`praproses_lengkap`. Kata test yang tak pernah ada di train (OOV): bila porsi tokennya > 10%,
test berbeda topik/gaya → TF-IDF karakter dan BERT lebih tahan daripada TF-IDF kata.

In [ ]:
phrase_counter = CountVectorizer(ngram_range=(2, 3), min_df=5, max_features=50000)
phrase_counts = phrase_counter.fit_transform(raw_train)
phrase_total = np.asarray((phrase_counts > 0).sum(axis=0)).ravel()
phrases = pd.DataFrame({"frasa": phrase_counter.get_feature_names_out(),
                        "teks yang memuat %": np.round(100 * phrase_total / len(train), 1)})
phrases = phrases.sort_values("teks yang memuat %", ascending=False)
print(phrases.head(12).to_string(index=False))

test_counter = CountVectorizer(token_pattern=r"(?u)\b\w{2,}\b")
test_counts = test_counter.fit_transform(raw_test)
test_vocabulary = test_counter.get_feature_names_out()
train_vocabulary = CountVectorizer(token_pattern=r"(?u)\b\w{2,}\b").fit(raw_train).vocabulary_
is_new = ~np.isin(test_vocabulary, list(train_vocabulary))
token_totals = np.asarray(test_counts.sum(axis=0)).ravel()
shown_new = is_new
if AMAN_NDA:
    shown_new = is_new & (token_totals >= 5)        # kata langka = jejak satu teks
new_order = np.argsort(-token_totals * shown_new)[:10]
new_order = new_order[shown_new[new_order]]
note(f"kata baru di test: {share(is_new)}% kosakata test, "
     f"{100 * token_totals[is_new].sum() / token_totals.sum():.1f}% token test | "
     f"contoh: {', '.join(test_vocabulary[new_order])}")

### 7e. Teks kembar
`kunci` = teks huruf kecil tanpa tanda baca, jadi kembar yang hanya beda spasi/tanda baca ikut
terhitung. Kembar beda target = label berisik (atau teks saja tidak cukup, butuh kolom tabel).
Test yang ada di train: pakai jawaban train untuk baris itu, dan pakai split berkelompok
(`groups = pd.factorize(teks)[0]`) supaya skor CV jujur.

In [ ]:
def text_key(texts):
    keys = texts.str.lower().str.replace(r"[\W_]+", " ", regex=True)
    return keys.str.strip()


train_key = text_key(raw_train)
test_key = text_key(raw_test)
filled = train_key != ""
targets_per_text = train[TARGET].groupby(train_key).transform("nunique")
conflict = filled & (targets_per_text > 1)
in_train = test_key.isin(set(train_key[filled])) & (test_key != "")
note(f"teks kembar di train: {int(train_key[filled].duplicated().sum())} baris | "
     f"kembar dengan target berbeda: {int(conflict.sum())} baris")
note(f"teks test yang juga ada di train: {int(in_train.sum())} dari {len(test)} "
     f"({share(in_train)}%)")
note(f"teks kosong: train {int((~filled).sum())}, test {int((test_key == '').sum())}")
conflict_rows = train.loc[conflict, TEXT_COLS + [TARGET]]
if not AMAN_NDA:
    print(conflict_rows.iloc[np.argsort(train_key[conflict].to_numpy(), kind="stable")].head(6))

## 8. Kekuatan tiap kolom sendirian terhadap target (dan cek bocor)
Pohon kecil dilatih dengan SATU kolom saja, 3-fold. Dibandingkan dengan `kolom acak` (pohon
yang sama, diberi angka acak). Ikut diuji: `id`, urutan baris, dan ciri teks (`teks: ...`).

- skor jauh di atas `kolom acak` → kolom berguna;
- skor hampir sempurna → **curiga bocor** (kolom dibuat dari target); cek ada/tidaknya di test;
- `id` atau `urutan baris` di atas `kolom acak` → data terurut menurut target/waktu: selidiki,
  dan jangan pakai id sebagai fitur kecuali polanya juga berlaku di test.

In [ ]:
def text_meta(texts):
    meta = pd.DataFrame(index=texts.index)
    letters = texts.str.count(r"[A-Za-z]").clip(lower=1)
    meta["teks: jumlah kata"] = texts.str.split().str.len()
    meta["teks: jumlah huruf"] = texts.str.len()
    meta["teks: rasio kapital"] = texts.str.count(r"[A-Z]") / letters
    meta["teks: tanda seru"] = texts.str.count("!")
    meta["teks: tanda tanya"] = texts.str.count(r"\?")
    meta["teks: jumlah angka"] = texts.str.count(r"\d+")
    meta["teks: emoji"] = texts.apply(emoji.emoji_count)
    meta["teks: kata unik / kata"] = texts.apply(unique_ratio)
    return meta


def unique_ratio(text):
    words = text.lower().split()
    return len(set(words)) / max(len(words), 1)


def as_numbers(values):
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(-999).to_numpy()
    return pd.factorize(values)[0]


meta_train = text_meta(raw_train)
meta_test = text_meta(raw_test)
meta_train.describe().round(2)

In [ ]:
if target_is_number:
    tree = DecisionTreeRegressor(max_depth=5, min_samples_leaf=20)
    SCORING, STRENGTH, folds = "r2", "r2", KFold(n_splits=3, shuffle=True, random_state=SEED)
else:
    tree = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, class_weight="balanced")
    folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    SCORING, STRENGTH = "f1_macro", "balanced_accuracy"
y = train[TARGET].to_numpy()
candidates = {ID_COL: train[ID_COL], "urutan baris": pd.Series(np.arange(len(train)))}
for column in NUM_COLS + CAT_COLS:
    candidates[column] = train[column]
for column in meta_train.columns:
    candidates[column] = meta_train[column]

noise = np.random.default_rng(SEED).normal(size=(len(train), 1))
base_score = cross_val_score(tree, noise, y, cv=folds, scoring=STRENGTH).mean()
strength_rows = [{"kolom": "kolom acak", "skor": round(base_score, 4)}]
for name, values in candidates.items():
    features = as_numbers(values).reshape(-1, 1)
    value = cross_val_score(tree, features, y, cv=folds, scoring=STRENGTH).mean()
    strength_rows.append({"kolom": name, "skor": round(value, 4)})
strength = pd.DataFrame(strength_rows).sort_values("skor", ascending=False)
strength["di atas kolom acak"] = (strength["skor"] - base_score).round(4)
strength.to_csv(f"{OUTPUT_FOLDER}/kekuatan_kolom.csv", index=False)
print("metrik:", STRENGTH)
strength

In [ ]:
figure, ax = plt.subplots(figsize=(13, 0.32 * len(strength) + 1.5))
colors = np.where(strength["kolom"] == "kolom acak", PALETTE[3], PALETTE[0])
ax.barh(strength["kolom"], strength["skor"], color=colors)
ax.axvline(base_score, color=PALETTE[3], linestyle="--")
ax.invert_yaxis()
ax.set_xlabel(f"{STRENGTH} dengan satu kolom saja (garis putus = kolom acak)")
show(figure, "08_kekuatan_kolom")

# pohon pada kolom acak bisa di bawah tebakan (r2 < 0), jadi bandingkan juga dengan tebakan
chance = 0
if not target_is_number:
    chance = 1 / train[TARGET].nunique()
beaten = strength["skor"] - max(base_score, chance) > 0.02
useful = strength[beaten & (strength["kolom"] != "kolom acak")]
note(f"kolom yang sendirian sudah mengalahkan kolom acak ({STRENGTH}): {useful['kolom'].tolist()}")
leaky = strength.loc[strength["skor"] > 0.95, "kolom"].tolist()
if leaky:
    note(f"CURIGA BOCOR (skor > 0,95 dengan satu kolom): {leaky}")
order_cols = useful.loc[useful["kolom"].isin([ID_COL, "urutan baris"]), "kolom"].tolist()
if order_cols:
    note(f"PERHATIAN: {order_cols} membawa informasi target, data kemungkinan terurut")

## 9. Train vs test: apa yang bergeser?
### 9a. Kolom angka, kategori, dan ciri teks
`geser` = statistik KS (angka, ciri teks) atau jarak sebaran (kategori); 0 = sama, 1 = beda total.
`geser` > 0,1 pada kolom yang kuat di bagian 8 = bahaya: skor CV akan lebih bagus daripada skor
test. Pilihan: buang kolom itu, atau buat validasi yang meniru test.

In [ ]:
shift_rows = []
for column in NUM_COLS:
    statistic = stats.ks_2samp(train[column].dropna(), test[column].dropna())[0]
    shift_rows.append({"kolom": column, "jenis": "angka", "geser": round(statistic, 3)})
for column in meta_train.columns:
    statistic = stats.ks_2samp(meta_train[column], meta_test[column])[0]
    shift_rows.append({"kolom": column, "jenis": "ciri teks", "geser": round(statistic, 3)})
for column in CAT_COLS:
    in_train_share = train[column].astype(str).value_counts(normalize=True)
    in_test_share = test[column].astype(str).value_counts(normalize=True)
    difference = in_train_share.subtract(in_test_share, fill_value=0)
    gap = difference.abs().sum() / 2
    shift_rows.append({"kolom": column, "jenis": "kategori", "geser": round(gap, 3)})
shift = pd.DataFrame(shift_rows).sort_values("geser", ascending=False)
shifted = shift.loc[shift["geser"] > 0.1, "kolom"].tolist()
note(f"kolom yang bergeser train -> test (geser > 0,1): {shifted}")
shift.head(15)

### 9b. Teks: adversarial validation
Model menebak "baris ini train atau test?" dari teksnya. AUC ≈ 0,5 = tidak bisa dibedakan (aman).
AUC > 0,65 = test beda gaya/topik/sumber: kata paling khas test menunjukkan bedanya apa.

In [ ]:
picked_train = raw_train.sample(min(len(train), SAMPLE_ROWS), random_state=SEED)
picked_test = raw_test.sample(min(len(test), SAMPLE_ROWS), random_state=SEED)
both = pd.concat([picked_train, picked_test], ignore_index=True)
is_test = np.r_[np.zeros(len(picked_train)), np.ones(len(picked_test))]
adversary_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=RARE_MIN, sublinear_tf=True)
adversary = make_pipeline(adversary_tfidf, LogisticRegression(C=1, max_iter=500))
adversary_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
proba = cross_val_predict(adversary, both, is_test, cv=adversary_folds, method="predict_proba")
adversary_auc = roc_auc_score(is_test, proba[:, 1])
adversary.fit(both, is_test)
adversary_words = adversary[0].get_feature_names_out()
order = np.argsort(adversary[-1].coef_[0])
note(f"adversarial AUC teks train vs test: {adversary_auc:.3f} (0,5 = sama, > 0,65 = test beda)")
note(f"kata paling khas test: {', '.join(adversary_words[order[-8:]])}")
note(f"kata paling khas train: {', '.join(adversary_words[order[:8]])}")

## 10. Teks ↔ angka
Korelasi Spearman antara ciri teks (baris) dan kolom angka + target angka (kolom). Contoh bacaan:
`jumlah kata` berkorelasi dengan kolom angka X → teks panjang cenderung X besar, jadi sebagian
informasi X sudah ada di teks.

In [ ]:
if corr_cols:
    cross = pd.DataFrame(index=meta_train.columns, columns=corr_cols, dtype=float)
    for meta_name in meta_train.columns:
        for column in corr_cols:
            cross.loc[meta_name, column] = stats.spearmanr(meta_train[meta_name], train[column],
                                                            nan_policy="omit")[0]
    cross = cross.fillna(0)
    figure, ax = plt.subplots(figsize=(13, 0.5 * len(cross) + 2))
    heatmap(ax, cross)
    show(figure, "10_teks_vs_angka")
    strength_pairs = cross.abs().stack()
    strongest = strength_pairs.idxmax()
    note(f"hubungan teks-angka terkuat: {strongest[0]} ~ {strongest[1]} "
         f"(r = {cross.loc[strongest[0], strongest[1]]:.2f})")

## 11. Sinyal ada di mana: teks saja vs tabel saja
Tiga model kecil, 3-fold yang sama, `SAMPLE_ROWS` baris: (a) TF-IDF teks, (b) pohon boosting pada
kolom angka + kategori, (c) pohon boosting pada ciri teks saja.

- tabel jauh di atas `kolom acak` → gabungkan: `stacking_teks_tabular` (OOF teks + kolom tabel);
- tabel ≈ `kolom acak` → kolom tabel tidak membantu, fokus ke teks (`baseline_oof`);
- ciri teks saja sudah tinggi → panjang/gaya tulisan membawa label: masukkan `meta_*` ke stacking.

In [ ]:
picked = train.sample(min(len(train), SAMPLE_ROWS), random_state=SEED).index
table_features = pd.DataFrame(index=train.index)
for column in NUM_COLS + CAT_COLS:
    table_features[column] = as_numbers(train[column])
if target_is_number:
    text_model, table_model = Ridge(alpha=1.0), HistGradientBoostingRegressor(random_state=SEED)
else:
    text_model = LogisticRegression(C=10, max_iter=1000, class_weight="balanced")
    table_model = HistGradientBoostingClassifier(random_state=SEED, class_weight="balanced")
text_pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                          text_model)
y_picked = train.loc[picked, TARGET].to_numpy()
signal_rows = []
sources = [("kolom acak", table_model, noise[:len(picked)]),
           ("teks saja (TF-IDF)", text_pipe, raw_train.loc[picked]),
           ("ciri teks saja (panjang, tanda baca)", table_model, meta_train.loc[picked])]
if NUM_COLS + CAT_COLS:
    sources.append(("tabel saja (angka + kategori)", table_model, table_features.loc[picked]))
for name, estimator, features in sources:
    scores = cross_val_score(estimator, features, y_picked, cv=folds, scoring=SCORING)
    signal_rows.append({"sumber": name, "skor": round(scores.mean(), 4)})
signal = pd.DataFrame(signal_rows)
for row in signal_rows:
    note(f"sinyal {row['sumber']}: {SCORING} {row['skor']}")
signal

## 12. Waktu (hanya bila `DATE_COL` diisi)
Test seluruhnya sesudah train → split acak terlalu optimistis, pakai split waktu (validasi =
bagian terbaru). Porsi target yang berubah dari bulan ke bulan = data bergeser terhadap waktu.

In [ ]:
if DATE_COL is not None:
    train_dates = pd.to_datetime(train[DATE_COL], errors="coerce")
    test_dates = pd.to_datetime(test[DATE_COL], errors="coerce")
    note(f"tanggal train {train_dates.min()} s/d {train_dates.max()} | "
         f"test {test_dates.min()} s/d {test_dates.max()} | "
         f"gagal dibaca: train {share(train_dates.isna())}%, test {share(test_dates.isna())}%")
    figure, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    train_months = train_dates.dt.to_period("M").astype(str)
    test_months = test_dates.dt.to_period("M").astype(str)
    month_counts = pd.crosstab(pd.concat([train_months, test_months]),
                               np.r_[["train"] * len(train), ["test"] * len(test)],
                               colnames=[""])
    month_counts.plot(ax=axes[0], marker="o")
    axes[0].set_xlabel("baris per bulan")
    mix = pd.crosstab(train_months, group, normalize="index")
    mix.plot(ax=axes[1], marker="o", color=PALETTE)
    axes[1].set_xlabel(f"porsi kelompok {TARGET} per bulan")
    show(figure, "12_waktu")
else:
    print("DATE_COL kosong: bagian waktu dilewati")

## Simpan temuan
`temuan.md` = semua baris bertanda `-` di atas, siap jadi bahan slide "what we found" dan dasar
keputusan split/model.

In [ ]:
with open(f"{OUTPUT_FOLDER}/temuan.md", "w", encoding="utf-8") as report_file:
    report_file.write("# Temuan EDA\n\n")
    for finding in findings:
        report_file.write(f"- {finding}\n")
print(len(findings), "temuan ditulis ke", f"{OUTPUT_FOLDER}/temuan.md")
print("\n- ".join(["# Temuan EDA"] + findings))   # ikut tersimpan di output notebook
print("file:", sorted(os.listdir(OUTPUT_FOLDER)))